In [6]:
import pandas as pd
import numpy as np
 
df = pd.read_csv("Team_Stats_Merge_2018_2025.csv")
target = df["made_playoffs"]
 
print("FEATURE SELECTION ANALYSIS")

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Target: made_playoffs (0={target.value_counts()[0]}, 1={target.value_counts()[1]})")

FEATURE SELECTION ANALYSIS
Dataset: 256 rows x 121 columns
Target: made_playoffs (0=148, 1=108)


### Outcome Leakage Variables

These variables directly encode win-loss outcomes. Playoff qualification is almost perfectly determined by wins, so including them would let the model memorize standings rather than learn structural performance patterns.

In [4]:
leakage_cols = ["tp_W", "tp_L", "tp_W-L%", "tp_PF", "tp_PA", "tp_PD", "tp_MoV"]

desc = {
    "tp_W": "Wins (direct determinant of playoff qualification)",
    "tp_L": "Losses (complement of wins)",
    "tp_W-L%": "Win pct (derived from W and L)",
    "tp_PF": "Points For (raw scoring total)",
    "tp_PA": "Points Against (raw opponent scoring)",
    "tp_PD": "Point Differential (PF - PA)",
    "tp_MoV": "Margin of Victory (PD / games)",
}

leakage_rows = []
for col in leakage_cols:
    r = df[col].corr(target)
    leakage_rows.append({
        "Variable": col,
        "Correlation": round(r, 3),
        "Description": desc[col],
    })

display(pd.DataFrame(leakage_rows).set_index("Variable"))

,Correlation,Description
Variable,,
tp_W,0.805,Wins (direct determinant of playoff qualificat...
tp_L,-0.797,Losses (complement of wins)
tp_W-L%,0.804,Win pct (derived from W and L)
tp_PF,0.623,Points For (raw scoring total)
tp_PA,-0.535,Points Against (raw opponent scoring)
tp_PD,0.729,Point Differential (PF - PA)
tp_MoV,0.730,Margin of Victory (PD / games)


The correlations are extremely high because these metrics directly determine the outcome we are predicting, so we are going to drop these 7 variables.

### Simple Rating System (SRS) Variables

SRS = Margin of Victory + Strength of Schedule, Since MoV is derived from point diferential, SRS inherits that leakage. OSRS and DSRS are components of SRS. tp_SoS measure schedule difficulty independently of team results, so this will be retained.

In [9]:
srs_cols = ["tp_SRS", "tp_OSRS", "tp_DSRS", "tp_SoS"]
srs_rows = []
for col in srs_cols:
    r = df[col].corr(target)
    action = "RETAIN (schedule context only)" if col == "tp_SoS" else "DROP (outcome-derived)"
    srs_rows.append({"Variable": col, "Correlation": round(r, 3), "Decision": action})
 
pd.DataFrame(srs_rows).set_index("Variable")

,Correlation,Decision
Variable,,
tp_SRS,0.711,DROP (outcome-derived)
tp_OSRS,0.605,DROP (outcome-derived)
tp_DSRS,0.503,DROP (outcome-derived)
tp_SoS,-0.256,RETAIN (schedule context only)


Decision: DROP SRS, OSRS, DSRS. RETAIN tp_SoS

### Games Plaed

Determining whether to keep or drop the seven games-played columns, as there is extremely minor differences in games played with each season.

In [10]:
games_cols = ["off_g", "def_g", "off_pass_G", "off_rush_G", "def_pass_G"]
if "def_rush_G" in df.columns:
    games_cols.append("def_rush_G")
if "def_adv_G" in df.columns:
    games_cols.append("def_adv_G")
 
games_rows = []
for col in games_cols:
    if col in df.columns:
        vals = df[col].value_counts()
        for v, count in vals.items():
            games_rows.append({
                "Column": col,
                "Value": int(v),
                "Count": count,
                "Pct": f"{count/len(df)*100:.1f}%"
            })
pd.DataFrame(games_rows)

,Column,Value,Count,Pct
0,off_g,17,158,61.7%
1,off_g,16,98,38.3%
2,def_g,17,158,61.7%
3,def_g,16,98,38.3%
4,off_pass_G,17,158,61.7%
5,off_pass_G,16,98,38.3%
6,off_rush_G,17,158,61.7%
7,off_rush_G,16,98,38.3%
8,def_pass_G,17,158,61.7%
9,def_pass_G,16,98,38.3%


Decision - Drop all as the values of 16 and 17 for all teams provide no discriminative signal for playoff prediction.

### Fourth Quarter Comebacks and Game Winning Drives

In [12]:
sit_rows = []
for col in ["off_pass_4QC", "off_pass_GWD"]:
    missing = df[col].isna().sum()
    pct = missing / len(df) * 100
    r = df[col].corr(target)
    sit_rows.append({
        "Variable": col,
        "Missing": missing,
        "Missing %": f"{pct:.1f}%",
        "Correlation": round(r, 3)
    })
 
display(pd.DataFrame(sit_rows).set_index("Variable"))
 
# Missing by season detail
sit_season_rows = []
for col in ["off_pass_4QC", "off_pass_GWD"]:
    for s in sorted(df["season"].unique()):
        n_miss = df[df["season"] == s][col].isna().sum()
        if n_miss > 0:
            sit_season_rows.append({"Variable": col, "Season": s, "Missing": n_miss})
 
display(pd.DataFrame(sit_season_rows))

,Missing,Missing %,Correlation
Variable,,,
off_pass_4QC,16,6.2%,0.257
off_pass_GWD,16,6.2%,0.343


,Variable,Season,Missing
0,off_pass_4QC,2019,2
1,off_pass_4QC,2020,4
2,off_pass_4QC,2021,5
3,off_pass_4QC,2022,1
4,off_pass_4QC,2023,1
5,off_pass_4QC,2024,1
6,off_pass_4QC,2025,2
7,off_pass_GWD,2019,2
8,off_pass_GWD,2020,4
9,off_pass_GWD,2021,5


Decision: Drropping both as they describe how wins happened, not why the team was competitive, 6.2% missing scattered across seasons, and misaligned with the efficiency-focused analytical framework.

### Tie's in the NFL tp_T

In [13]:
ties_dist = df["tp_T"].value_counts(dropna=False).reset_index()
ties_dist.columns = ["Value", "Count"]
display(ties_dist)
 
ties_season = []
for s in sorted(df["season"].unique()):
    subset = df[df["season"] == s]
    ties_season.append({
        "Season": s,
        "Null": subset["tp_T"].isna().sum(),
        "Zero": (subset["tp_T"] == 0).sum(),
        "One": (subset["tp_T"] == 1).sum(),
    })
pd.DataFrame(ties_season).set_index("Season")
 

,Value,Count
0,0.0,176
1,NaN,64
2,1.0,16


,Null,Zero,One
Season,,,
2018,0,28,4
2019,0,30,2
2020,0,30,2
2021,0,30,2
2022,0,28,4
2023,32,0,0
2024,32,0,0
2025,0,30,2


Decision: Drop — 176 zeros, 16 ones, 64 nulls in 2023-2024

### Multicollinearity Analysis 

Identifying all feature pairs with |r| > 0.90 to find clusters of near-identical variables that should be reduced

In [14]:
# Get all numeric columns (excluding keys and target)
numeric_cols = [c for c in df.select_dtypes(include=["int64", "float64"]).columns
                if c not in ["season", "made_playoffs"]]
 
corr_matrix = df[numeric_cols].corr()
 
# Find highly correlated pairs
high_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.90:
            c1, c2 = corr_matrix.columns[i], corr_matrix.columns[j]
            high_pairs.append({"Feature A": c1, "Feature B": c2, "Correlation": round(r, 3)})
 
high_pairs_df = pd.DataFrame(high_pairs).sort_values("Correlation", key=abs, ascending=False)
high_pairs_df.reset_index(drop=True)

,Feature A,Feature B,Correlation
0,tp_PD,tp_MoV,1.000
1,tp_PA,def_pa,1.000
2,tp_PF,off_pts,1.000
3,off_g,off_pass_G,1.000
4,off_g,off_rush_G,1.000
...,...,...,...
78,off_pts,off_exp,0.908
79,def_pass_NY/A,def_pass_ANY/A,0.907
80,off_yds_per_play,off_pass_ANY/A,0.906
81,off_pass_AY/A,off_pass_NY/A,0.906


Now, with these 83 pairs, we can further examine which features to drop.

I will pair these features by their cluster/category they pertain to, such as offensive passing, volume vs efficieny for offense, defensive passing, and rushing volume

In [15]:
off_pass_efficiency = ["off_pass_Y/A", "off_pass_AY/A", "off_pass_NY/A",
                       "off_pass_ANY/A", "off_pass_EXP", "off_pass_Rate"]
 
# Intercorrelation matrix
eff_corr = df[off_pass_efficiency].corr().round(3)
display(eff_corr)
 
# Target correlations - choosing representative
eff_target_rows = []
for col in off_pass_efficiency:
    r = abs(df[col].corr(target))
    eff_target_rows.append({
        "Feature": col,
        "|r| with playoffs": round(r, 3),
        "Decision": "RETAIN" if col == "off_pass_ANY/A" else "DROP"
    })
pd.DataFrame(eff_target_rows).set_index("Feature")

,off_pass_Y/A,off_pass_AY/A,off_pass_NY/A,off_pass_ANY/A,off_pass_EXP,off_pass_Rate
off_pass_Y/A,1.000,0.923,0.940,0.882,0.812,0.831
off_pass_AY/A,0.923,1.000,0.906,0.972,0.891,0.963
off_pass_NY/A,0.940,0.906,1.000,0.943,0.909,0.836
off_pass_ANY/A,0.882,0.972,0.943,1.000,0.945,0.947
off_pass_EXP,0.812,0.891,0.909,0.945,1.000,0.889
off_pass_Rate,0.831,0.963,0.836,0.947,0.889,1.000


,|r| with playoffs,Decision
Feature,,
off_pass_Y/A,0.420,DROP
off_pass_AY/A,0.533,DROP
off_pass_NY/A,0.490,DROP
off_pass_ANY/A,0.563,RETAIN
off_pass_EXP,0.549,DROP
off_pass_Rate,0.560,DROP


Decision: Keep off_pass_ANY/A because it has the strongest relationsip with playoff qualification.
* you may notice off_pass_Rate is at 0.560, however I chose ANY/A because it also has the most comprehensive metric from a football perspective because it adjusts for sacks, interceptions, and touchdowns all in one number, while the others only adjust for some of those factors.

In [18]:
 # Volume vs efficiency for offense
off_volume = ["off_pass_Yds", "off_pass_Att", "off_pass_Cmp", "off_pass_TD",
              "off_pass_Lng", "off_pass_Y/C", "off_pass_Y/G", "off_pass_Yds.1"]
 
vol_rows = []
for col in off_volume:
    if col in df.columns:
        r = abs(df[col].corr(target))
        vol_rows.append({"Feature": col, "|r| with playoffs": round(r, 3)})
 
pd.DataFrame(vol_rows).set_index("Feature")

,|r| with playoffs
Feature,
off_pass_Yds,0.333
off_pass_Att,0.021
off_pass_Cmp,0.146
off_pass_TD,0.487
off_pass_Lng,0.053
off_pass_Y/C,0.264
off_pass_Y/G,0.312
off_pass_Yds.1,0.414


off_pass_Cmp%, off_pass_TD%, off_pass_Int%, and off_pass_Rate are NOT included in this drop list even though they are passing metrics. These measure DISTINCT dimensions of passing performance (accuracy, scoring rate, mistake rate, and a composite rating) and are NOT highly correlated with each other or with off_pass_ANY/A at the |r| > 0.90 threshold. They each capture unique information, unlike the Y/A variants which are all slight mathematical adjustments of the same yards-per-attempt calculation. These are retained alongside ANY/A.

In [22]:
# Defensive passing cluster
def_pass_all = ["def_pass_Y/A", "def_pass_AY/A", "def_pass_NY/A",
                "def_pass_ANY/A", "def_pass_EXP", "def_pass_Rate"]
 
def_target_rows = []
for col in def_pass_all:
    if col in df.columns:
        r = abs(df[col].corr(target))
        def_target_rows.append({
            "Feature": col,
            "|r| with playoffs": round(r, 3)
        })

pd.DataFrame(def_target_rows).set_index("Feature")

,|r| with playoffs
Feature,
def_pass_Y/A,0.389
def_pass_AY/A,0.452
def_pass_NY/A,0.408
def_pass_ANY/A,0.452
def_pass_EXP,0.415
def_pass_Rate,0.437


Same logic as offensive passing: the Y/A variant cluster (Y/A, AY/A, NY/A, ANY/A, EXP, Rate) are near-perfect linear combinations of each other. I chose def_pass_ANY/A and dropped the rest.

def_pass_Cmp%, def_pass_TD%, def_pass_Int%, and def_pass_Rate are RETAINED because they measure distinct dimensions (completion accuracy allowed, scoring rate allowed, interception rate, and composite rating) that are not highly correlated with each other at |r| > 0.90. Additionally, defensive-specific metrics like def_pass_Sk, def_pass_QBHits, def_pass_TFL, and def_pass_Sk% are retained because they measure pass rush pressure, which has no direct equivalent in the offensive passing stats.

In [23]:
# Rushing Efficieny Metrics we are retaining

rush_efficiency = ["off_rush_Y/A", "def_rush_Y/A"]
rush_eff_rows = []
for col in rush_efficiency:
    if col in df.columns:
        r = abs(df[col].corr(target))
        rush_eff_rows.append({"Feature": col, "|r| with playoffs": round(r, 3)})

The above metrics measures yards per rush attempt, which normalizes how often a team runs. Similar to the ANY/A for passing.

In [32]:
# Rushing volume/redundant metrics dropped


rush_vol = ["off_rush_Lng", "off_rush_Y/G", "off_rush_Fmb", "def_rush_Y/G"]
rush_eff_corr = df[rush_vol].corr()
display(rush_eff_corr.round(3))

results = []

for col in rush_vol:
    if col in df.columns:
        r = abs(df[col].corr(target))
        results.append({"Column": col, "|r|": round(r, 3)})

# Convert to DataFrame and display
pd.DataFrame(results)

,off_rush_Lng,off_rush_Y/G,off_rush_Fmb,def_rush_Y/G
off_rush_Lng,1.000,0.280,-0.019,0.004
off_rush_Y/G,0.280,1.000,0.095,-0.187
off_rush_Fmb,-0.019,0.095,1.000,0.049
def_rush_Y/G,0.004,-0.187,0.049,1.000


,Column,|r|
0,off_rush_Lng,0.033
1,off_rush_Y/G,0.322
2,off_rush_Fmb,0.134
3,def_rush_Y/G,0.409


In [28]:
# Show intercorrelation between Y/G and Y/A to further confirm redundancy

redundancy_rows = []
if "off_rush_Y/G" in df.columns and "off_rush_Y/A" in df.columns:
    r = df["off_rush_Y/G"].corr(df["off_rush_Y/A"])
    redundancy_rows.append({"Pair": "off_rush_Y/G vs off_rush_Y/A", "Correlation": round(r, 3)})
if "def_rush_Y/G" in df.columns and "def_rush_Y/A" in df.columns:
    r = df["def_rush_Y/G"].corr(df["def_rush_Y/A"])
    redundancy_rows.append({"Pair": "def_rush_Y/G vs def_rush_Y/A", "Correlation": round(r, 3)})
 
pd.DataFrame(redundancy_rows).set_index("Pair")

,Correlation
Pair,
off_rush_Y/G vs off_rush_Y/A,0.811
def_rush_Y/G vs def_rush_Y/A,0.810


Decision: dropping the 4 metrics above because:
* off_rush_Lng is a single-play stat not a team indicator
* off_rush_Y/G is a volume stat not efficiency per attempt
* Fmb already captured in off_turnovers and off_fumbles
* def_rush_Y/G is also a volume stat not efficiency per attempt

### Defense Advance Duplicate Check

Comparing def_adv_ columns against existing def_pass_ columns to identify identical data stored under different names.

In [34]:
# identify all def_adv columns
all_adv_cols = [c for c in df.columns if c.startswith("def_adv_")]
len(all_adv_cols)

18

In [35]:
null_cols = []
complete_cols = []
 
completeness_rows = []
for col in all_adv_cols:
    non_null = df[col].notna().sum()
    missing = df[col].isna().sum()
    pct = non_null / len(df) * 100
    if missing == len(df):
        status = "ENTIRELY NULL - DROP"
        null_cols.append(col)
    else:
        status = "has data"
        complete_cols.append(col)
    completeness_rows.append({
        "Column": col,
        "Non-Null": non_null,
        "Missing": missing,
        "Complete %": f"{pct:.1f}%",
        "Status": status
    })
pd.DataFrame(completeness_rows).set_index("Column")

,Non-Null,Missing,Complete %,Status
Column,,,,
def_adv_G,256,0,100.0%,has data
def_adv_Att,256,0,100.0%,has data
def_adv_Cmp,256,0,100.0%,has data
def_adv_Yds,256,0,100.0%,has data
def_adv_TD,256,0,100.0%,has data
def_adv_DADOT,256,0,100.0%,has data
def_adv_Air,256,0,100.0%,has data
def_adv_YAC,256,0,100.0%,has data
def_adv_Bltz,256,0,100.0%,has data


There are 4 columns without any data at all.

**Checking for duplicate columns**

Comparing the def_adv columns against the non adv defense columns to find exact or very close dupes

In [40]:
existing_cols = [c for c in df.select_dtypes(include=["int64", "float64"]).columns
                 if not c.startswith("def_adv_") and c not in ["season", "made_playoffs"]]
 
duplicate_cols = []
unique_cols = []
 
dup_rows = []
for adv_col in complete_cols:
    best_corr = 0
    best_match = ""
    is_exact = False
 
    for exist_col in existing_cols:
        r = abs(df[adv_col].corr(df[exist_col]))
        if r > best_corr:
            best_corr = r
            best_match = exist_col
            if r > 0.99:
                is_exact = (df[adv_col] == df[exist_col]).all()
 
    dup_rows.append({
        "def_adv_ Column": adv_col,
        "Best Match": best_match,
        "Correlation": round(best_corr, 4)
    })
pd.DataFrame(dup_rows).set_index("def_adv_ Column")

,Best Match,Correlation
def_adv_ Column,,
def_adv_G,off_g,1.0000
def_adv_Att,def_pass_Att,1.0000
def_adv_Cmp,def_pass_Cmp,1.0000
def_adv_Yds,def_pass_Yds,1.0000
def_adv_TD,def_pass_TD,1.0000
def_adv_DADOT,def_pass_Y/C,0.6259
def_adv_Air,def_pass_Y/G,0.7058
def_adv_YAC,def_pass_Cmp,0.6187
def_adv_Bltz,def_pass_Att,0.3624


Here we gather that there 6 duplicates to drop here with the corr = 1.0000, and we will keep the other unique columns for further evaluation.

After removing duplicates and nulls, there are 8 unique def_adv metrics remaining. Now we will evaluate each for predictive signalling and redundancy with existing features.

description of each advaned metric:
* def_adv_DADOT: Depth of target (how deep passes are thrown)
* def_adv_Air: Air yards allowed (pre-catch passing yards)
* def_adv_YAC: Yards after catch allowed (post-catch yards)
* def_adv_Bltz: Total blitzes called
* def_adv_Hrry: Hurries generated
* def_adv_QBKD: QB knockdowns
* def_adv_Prss: Total pressures (hurries + knockdowns + sacks)
* def_adv_MTkl: Missed tackles

In [45]:
unique_adv = ["def_adv_DADOT", "def_adv_Air", "def_adv_YAC", "def_adv_Bltz",
              "def_adv_Hrry", "def_adv_QBKD", "def_adv_Prss", "def_adv_MTkl"]
 
# Target correlations
target_rows = []
for col in unique_adv:
    r = df[col].corr(target)
    target_rows.append({
        "Column": col,
        "Correlation": round(r, 3)
    })
pd.DataFrame(target_rows).set_index("Column")

,Correlation
Column,
def_adv_DADOT,0.008
def_adv_Air,0.342
def_adv_YAC,0.295
def_adv_Bltz,0.081
def_adv_Hrry,0.125
def_adv_QBKD,0.302
def_adv_Prss,0.348
def_adv_MTkl,-0.220


Now checking these advanced stats to see if any of them are redundant with each other.

In [47]:
# Intercorrelation among unique advanced stats

adv_corr = df[unique_adv].corr()
adv_corr.round(2)

,def_adv_DADOT,def_adv_Air,def_adv_YAC,def_adv_Bltz,def_adv_Hrry,def_adv_QBKD,def_adv_Prss,def_adv_MTkl
def_adv_DADOT,1.00,0.43,-0.28,0.01,0.09,-0.03,0.02,-0.10
def_adv_Air,0.43,1.00,0.32,0.18,0.19,0.20,0.22,-0.06
def_adv_YAC,-0.28,0.32,1.00,0.19,0.14,0.24,0.22,0.14
def_adv_Bltz,0.01,0.18,0.19,1.00,0.20,0.31,0.35,0.07
def_adv_Hrry,0.09,0.19,0.14,0.20,1.00,0.05,0.73,0.24
def_adv_QBKD,-0.03,0.20,0.24,0.31,0.05,1.00,0.63,-0.09
def_adv_Prss,0.02,0.22,0.22,0.35,0.73,0.63,1.00,0.04
def_adv_MTkl,-0.10,-0.06,0.14,0.07,0.24,-0.09,0.04,1.00


The correlation matrix above shows every pairs correlation and we can notice that def_adv_Hrry correlated with def_adv_Prss at r=0.73 telling us that keeping both may be somewhat redundant. If we can only keep one, pressures is the better choice because it's the broader metric that has to do with -- pressures = hurries + knockdowns + sacks combined.

In [48]:
 # Cross-correlation with existing retained features

existing_def_features = ["def_pass_Sk", "def_pass_Sk%", "def_pass_QBHits",
                         "def_pass_TFL", "def_pass_ANY/A", "def_pass_Rate",
                         "def_pass_Cmp%", "def_score_pct", "def_yds_per_play",
                         "def_turnovers_forced"]
 
cross_rows = []
for adv_col in unique_adv:
    max_r = 0
    max_partner = ""
    for exist_col in existing_def_features:
        if exist_col in df.columns:
            r = abs(df[adv_col].corr(df[exist_col]))
            if r > max_r:
                max_r = r
                max_partner = exist_col
    cross_rows.append({
        "Adv Column": adv_col,
        "Most Correlated Existing": max_partner,
        "Correlation": round(max_r, 3)
    })

pd.DataFrame(cross_rows).set_index("Adv Column")

,Most Correlated Existing,Correlation
Adv Column,,
def_adv_DADOT,def_pass_Cmp%,0.376
def_adv_Air,def_yds_per_play,0.372
def_adv_YAC,def_pass_Cmp%,0.292
def_adv_Bltz,def_pass_QBHits,0.306
def_adv_Hrry,def_turnovers_forced,0.169
def_adv_QBKD,def_pass_QBHits,0.795
def_adv_Prss,def_pass_QBHits,0.770
def_adv_MTkl,def_score_pct,0.269


We can observe the correlation above. One thing to note is how def_adv_QBKD and def_pass_QBHits correlate at r=0.795. This means knockdowns and hits are measuring nearly the same thing, so keeping both add little new information. Looking at this on the other end of the spectrum we can note how def_adv_Air correlates at r=0.372 which means air yards capture something genuinely new that no existing feature already measures, so that's why for example we keep that adv metric instead and drop QBKD.

Missing Values Analysis

In [56]:
# Scanning entire dataset to find missing values

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing_overview = pd.DataFrame([{
    "Columns with Missing": len(missing),
    "Overall Completeness": f"{(1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.2f}%"
}], index=["Dataset"])
display(missing_overview)

missing_rows = []
for col, count in missing.items():
    pct = count / len(df) * 100
    missing_rows.append({
        "Column": col,
        "Missing Count": count,
        "Missing %": f"{pct:.1f}%",
        "Status": "Entirely null" if count == 256 else "Partial"
    })
pd.DataFrame(missing_rows).set_index("Column")



,Columns with Missing,Overall Completeness
Dataset,9,96.38%


,Missing Count,Missing %,Status
Column,,,
def_adv_Prss%,256,100.0%,Entirely null
def_adv_Hrry%,256,100.0%,Entirely null
def_adv_QBKD%,256,100.0%,Entirely null
def_adv_Bltz%,256,100.0%,Entirely null
tp_T,64,25.0%,Partial
off_pass_GWD,16,6.2%,Partial
off_pass_4QC,16,6.2%,Partial
def_pass_Int%,1,0.4%,Partial
def_pass_Int,1,0.4%,Partial


In [59]:
# Season-level detail for partial missingness
season_missing_rows = []
for col, count in missing.items():
    if count < 256:
        for s in sorted(df["season"].unique()):
            n_miss = df[df["season"] == s][col].isna().sum()
            if n_miss > 0:
                season_missing_rows.append({"Column": col, "Season": s, "Missing": n_miss})
 
if season_missing_rows:
    display(pd.DataFrame(season_missing_rows))

,Column,Season,Missing
0,tp_T,2023,32
1,tp_T,2024,32
2,off_pass_GWD,2019,2
3,off_pass_GWD,2020,4
4,off_pass_GWD,2021,5
5,off_pass_GWD,2022,1
6,off_pass_GWD,2023,1
7,off_pass_GWD,2024,1
8,off_pass_GWD,2025,2
9,off_pass_4QC,2019,2
